# Première exploration de `application_train.csv`

Cette première exploration vérifie uniquement la structure générale du jeu de données, l'identifiant `SK_ID_CURR`, la cible `TARGET`, les valeurs manquantes et les doublons.

In [10]:
from pathlib import Path

import pandas as pd

# ---------- Chargement des données ----------
DATA_PATH = Path("../data/raw/application_train.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

## Dimensions et aperçu

In [11]:
print(f"Nombre de lignes : {df.shape[0]:,}")
print(f"Nombre de colonnes : {df.shape[1]}")
df.head()

Nombre de lignes : 307,511
Nombre de colonnes : 122


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## Vérification de `SK_ID_CURR`

Un identifiant valide doit être présent, sans valeur manquante et unique pour chaque ligne.

In [13]:
ID_COLUMN = "SK_ID_CURR"

if ID_COLUMN not in df.columns:
    raise KeyError(f"Colonne obligatoire absente : {ID_COLUMN}")

verification_id = pd.Series(
    {
        "valeurs_manquantes": int(df[ID_COLUMN].isna().sum()),
        "valeurs_uniques": int(df[ID_COLUMN].nunique()),
        "identifiants_dupliques": int(df[ID_COLUMN].duplicated().sum()),
        "identifiant_unique_par_ligne": bool(df[ID_COLUMN].is_unique),
    },
    name=ID_COLUMN,
)
verification_id.to_frame("résultat")

,résultat
valeurs_manquantes,0
valeurs_uniques,307511
identifiants_dupliques,0
identifiant_unique_par_ligne,True


## Analyse de `TARGET`

La cible est contrôlée puis sa répartition est présentée en effectifs et en pourcentages.

In [14]:
TARGET_COLUMN = "TARGET"

if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Colonne obligatoire absente : {TARGET_COLUMN}")

effectifs = df[TARGET_COLUMN].value_counts(dropna=False).sort_index()
repartition_target = pd.DataFrame(
    {
        "effectif": effectifs,
        "pourcentage": (effectifs / len(df) * 100).round(2),
    }
)
repartition_target.index.name = TARGET_COLUMN
repartition_target

,effectif,pourcentage
TARGET,,
0,282686,91.93
1,24825,8.07


## Valeurs manquantes

Le tableau affiche uniquement les colonnes qui contiennent au moins une valeur manquante.

In [15]:
nombre_manquantes = df.isna().sum()
valeurs_manquantes = pd.DataFrame(
    {
        "nombre de valeurs manquantes": nombre_manquantes,
        "pourcentage": (nombre_manquantes / len(df) * 100).round(2),
    }
)
valeurs_manquantes = valeurs_manquantes.loc[valeurs_manquantes["nombre de valeurs manquantes"] > 0]
valeurs_manquantes = valeurs_manquantes.sort_values("pourcentage", ascending=False)

print(f"Colonnes avec des valeurs manquantes : {len(valeurs_manquantes)} / {df.shape[1]}")
valeurs_manquantes

Colonnes avec des valeurs manquantes : 67 / 122


,nombre de valeurs manquantes,pourcentage
COMMONAREA_MEDI,214865,69.87
COMMONAREA_MODE,214865,69.87
COMMONAREA_AVG,214865,69.87
NONLIVINGAPARTMENTS_MODE,213514,69.43
NONLIVINGAPARTMENTS_MEDI,213514,69.43
...,...,...
EXT_SOURCE_2,660,0.21
AMT_GOODS_PRICE,278,0.09
AMT_ANNUITY,12,0.00
CNT_FAM_MEMBERS,2,0.00


## Doublons

Cette vérification recherche les lignes entièrement identiques.

In [16]:
nombre_doublons = int(df.duplicated().sum())
pourcentage_doublons = nombre_doublons / len(df) * 100

print(f"Nombre de lignes dupliquées : {nombre_doublons:,}")
print(f"Pourcentage de lignes dupliquées : {pourcentage_doublons:.2f} %")

Nombre de lignes dupliquées : 0
Pourcentage de lignes dupliquées : 0.00 %


## Types de données

In [12]:
types_par_colonne = df.dtypes.rename("type").to_frame()
display(types_par_colonne)

print("Résumé par type :")
display(df.dtypes.value_counts().rename("nombre_de_colonnes").to_frame())

,type
SK_ID_CURR,int64
TARGET,int64
NAME_CONTRACT_TYPE,str
CODE_GENDER,str
FLAG_OWN_CAR,str
...,...
AMT_REQ_CREDIT_BUREAU_DAY,float64
AMT_REQ_CREDIT_BUREAU_WEEK,float64
AMT_REQ_CREDIT_BUREAU_MON,float64
AMT_REQ_CREDIT_BUREAU_QRT,float64


Résumé par type :


,nombre_de_colonnes
float64,65
int64,41
str,16


In [18]:
# ---------- Détail des colonnes textuelles ----------
colonnes_texte = df.select_dtypes(include=["object", "string"]).columns

informations_colonnes_texte = pd.DataFrame(
    {
        "type": df[colonnes_texte].dtypes.astype(str),
        "nombre de valeurs uniques": df[colonnes_texte].nunique(dropna=True),
        "aperçu des valeurs": [
            df[colonne].dropna().unique()[:5].tolist()
            for colonne in colonnes_texte
        ],
    }
)
informations_colonnes_texte.index.name = "colonne"
informations_colonnes_texte

,type,nombre de valeurs uniques,aperçu des valeurs
colonne,,,
NAME_CONTRACT_TYPE,str,2,"[Cash loans, Revolving loans]"
CODE_GENDER,str,3,"[M, F, XNA]"
FLAG_OWN_CAR,str,2,"[N, Y]"
FLAG_OWN_REALTY,str,2,"[Y, N]"
NAME_TYPE_SUITE,str,7,"[Unaccompanied, Family, Spouse, partner, Child..."
NAME_INCOME_TYPE,str,8,"[Working, State servant, Commercial associate,..."
NAME_EDUCATION_TYPE,str,5,"[Secondary / secondary special, Higher educati..."
NAME_FAMILY_STATUS,str,6,"[Single / not married, Married, Civil marriage..."
NAME_HOUSING_TYPE,str,6,"[House / apartment, Rented apartment, With par..."


In [19]:
# ---------- Modalités de certaines colonnes textuelles ----------
colonnes_a_detailler = [
    "ORGANIZATION_TYPE",
    "OCCUPATION_TYPE",
    "NAME_FAMILY_STATUS",
]

colonnes_absentes = [
    colonne for colonne in colonnes_a_detailler if colonne not in df.columns
]
if colonnes_absentes:
    raise KeyError(f"Colonnes absentes : {colonnes_absentes}")

for colonne in colonnes_a_detailler:
    print(f"{colonne} :")
    modalites = (
        df[colonne]
        .value_counts(dropna=False)
        .rename_axis("valeur")
        .reset_index(name="effectif")
    )
    display(modalites)

ORGANIZATION_TYPE :


,valeur,effectif
0,Business Entity Type 3,67992
1,XNA,55374
2,Self-employed,38412
3,Other,16683
4,Medicine,11193
5,Business Entity Type 2,10553
6,Government,10404
7,School,8893
8,Trade: type 7,7831
9,Kindergarten,6880


OCCUPATION_TYPE :


,valeur,effectif
0,NaN,96391
1,Laborers,55186
2,Sales staff,32102
3,Core staff,27570
4,Managers,21371
5,Drivers,18603
6,High skill tech staff,11380
7,Accountants,9813
8,Medicine staff,8537
9,Security staff,6721


NAME_FAMILY_STATUS :


,valeur,effectif
0,Married,196432
1,Single / not married,45444
2,Civil marriage,29775
3,Separated,19770
4,Widow,16088
5,Unknown,2


## Encodage des colonnes textuelles binaires

Les colonnes textuelles qui possèdent exactement deux catégories sont encodées en `0` et `1`. Une copie du DataFrame est utilisée pour conserver les données initiales dans `df`.

In [20]:
# ---------- Encodage binaire ----------
df_encoded = df.copy()

colonnes_texte = df.select_dtypes(include=["object", "string"]).columns
colonnes_binaires = [
    colonne
    for colonne in colonnes_texte
    if df[colonne].nunique(dropna=True) == 2
]

correspondances = []

for colonne in colonnes_binaires:
    categories = sorted(df[colonne].dropna().unique().tolist())
    correspondance = {categories[0]: 0, categories[1]: 1}
    df_encoded[colonne] = df_encoded[colonne].map(correspondance).astype("Int8")

    correspondances.extend(
        {"colonne": colonne, "catégorie": categorie, "valeur encodée": valeur}
        for categorie, valeur in correspondance.items()
    )

print(f"Colonnes textuelles binaires encodées : {len(colonnes_binaires)}")
display(pd.DataFrame(correspondances))

Colonnes textuelles binaires encodées : 4


,colonne,catégorie,valeur encodée
0,NAME_CONTRACT_TYPE,Cash loans,0
1,NAME_CONTRACT_TYPE,Revolving loans,1
2,FLAG_OWN_CAR,N,0
3,FLAG_OWN_CAR,Y,1
4,FLAG_OWN_REALTY,N,0
5,FLAG_OWN_REALTY,Y,1
6,EMERGENCYSTATE_MODE,No,0
7,EMERGENCYSTATE_MODE,Yes,1


## One-hot encoding des autres colonnes textuelles

Les colonnes qui sont encore textuelles après l'encodage binaire sont transformées en colonnes indicatrices. Les valeurs manquantes disposent d'un indicateur dédié lorsqu'elles sont présentes.

In [21]:
# ---------- One-hot encoding ----------
colonnes_texte_restantes = df_encoded.select_dtypes(
    include=["object", "string"]
).columns.tolist()
colonnes_avec_valeurs_manquantes = [
    colonne
    for colonne in colonnes_texte_restantes
    if df_encoded[colonne].isna().any()
]

df_one_hot = pd.get_dummies(
    df_encoded,
    columns=colonnes_texte_restantes,
    prefix_sep="__",
    dtype="int8",
)

for colonne in colonnes_avec_valeurs_manquantes:
    df_one_hot[f"{colonne}__MANQUANT"] = (
        df_encoded[colonne].isna().astype("int8")
    )

print(f"Colonnes textuelles encodées : {len(colonnes_texte_restantes)}")
print(f"Dimensions avant le one-hot encoding : {df_encoded.shape}")
print(f"Dimensions après le one-hot encoding : {df_one_hot.shape}")
colonnes_texte_restantes

Colonnes textuelles encodées : 12
Dimensions avant le one-hot encoding : (307511, 122)
Dimensions après le one-hot encoding : (307511, 247)


['CODE_GENDER',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE']